# 🌧️ RainGuard AI - Eksplorasi Data & Visualisasi

Buku catatan (Notebook) Jupyter ini digunakan untuk mengeksplorasi data cuaca sintetis yang telah dibuat dan memvisualisasikan wawasan tentang pola curah hujan, serta relasinya dengan fitur cuaca lain seperti suhu dan kelembapan.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Mengamankan path impor supaya bisa mendapatkan modul lokal
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.preprocess import load_and_preprocess_data

### 1. Memuat dan Memeriksa Dataset
Kita akan menggunakan _preprocessing pipeline_ kita untuk memuat `data/rainfall_dataset.csv`. Pipeline ini juga sudah menambahkan fitur ekstra seperti `log_rainfall` dan indeks waktu (day_of_year, month).

In [ ]:
df = load_and_preprocess_data('../data/rainfall_dataset.csv')

print(f"Total baris data: {len(df)}")
print(f"Daftar Kolom: {list(df.columns)}")
df.head()

### 2. Cuaca Historis Berdasarkan Wilayah
Melihat sebaran ekstrimitas kejadian hujan (boxplot) antara beberapa kota besar.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='region', y='rainfall', palette='Blues')
plt.title('Sebaran Intensitas Curah Hujan per Wilayah')
plt.ylabel('Curah Hujan (mm)')
plt.xlabel('Wilayah')
plt.show()

### 3. Matriks Korelasi (Suhu vs Kelembapan vs Curah Hujan)
Apakah semakin lembap akan semakin tinggi curah hujannya?

In [ ]:
plt.figure(figsize=(8, 6))
# Menghitung korelasi metrik numerik saja
numerik_cols = ['temperature', 'humidity', 'rainfall']
corr_matrix = df[numerik_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriks Korelasi Cuaca')
plt.show()

### 4. Korelasi Berbasis Scatter
Memvisualisasikan keterkaitan visual antara suhu dan kelembapan, dengan titik besar dan warna pekat menandakan curah hujan tinggi.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df.sample(1000, random_state=42),
                x='temperature', y='humidity', hue='rainfall', 
                size='rainfall', sizes=(20, 300), palette='YlGnBu')

plt.title('Suhu vs Kelembapan terhadap Curah Hujan (Sample 1000)')
plt.xlabel('Suhu (°C)')
plt.ylabel('Kelembapan (%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.show()

### 5. Meninjau Tren Musiman Tahunan di Jakarta
Membongkar plot garis waktu rata-rata curah hujan bulanan di ibukota untuk melihat ada/tidaknya tren musim hujan sistematis di ujung dan awal tahun.

In [ ]:
jakarta_df = df[df['region'] == 'Jakarta'].copy()
jakarta_df = jakarta_df.set_index('date')
# Mengelompokkan ulang ke rata-rata curah hujan per bulan
jakarta_df_monthly = jakarta_df['rainfall'].resample('B').mean()

plt.figure(figsize=(15, 6))
plt.plot(jakarta_df_monthly.index, jakarta_df_monthly.values, marker='o', linestyle='-', color='b')
plt.title('Tren Rata-rata Curah Hujan Bulanan di Jakarta')
plt.xlabel('Waktu')
plt.ylabel('Rata-rata Curah Hujan Harian (mm)')
plt.grid(True)
plt.tight_layout()
plt.show()